# Chapter 36: constructed implementation shortfall

Use the same request for 100 shares, two 40-share fills and a 20-share remainder as Chapter 35. Decision price is 100, arrival price 101 and cutoff price 103. All amounts are USD; positive implementation shortfall means cost.

Check direct shortfall against delay cost 100, realized execution cost, unfilled cost 40 and explicit fees. The discrepancy should be within floating tolerance. Save constructed_is.json and one four-component PNG under RUN_DIR. This fresh mechanism neither runs the complete Almgren–Chriss scheduler nor calibrates its parameters. The execution engine's proportional coefficients are not automatically continuous or discrete AC eta/gamma.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). This stage delivers language views; portable dependencies and execution are checked in the later isolated cold start.

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.chdir(ROOT)
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch36_implementation_shortfall_constructed_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Default offline teaching; current empirical French execution SKIPPED (permissions unresolved).")
print("Run directory:", RUN_DIR)


In [ ]:
import datetime as dt
from dataclasses import asdict
import json
import matplotlib.pyplot as plt
from quant_course.execution_costs import ExecutionConfig, ParentOrder, ExecutionSlot, execute_order, ExecutionPortfolioLedger
start=dt.datetime(2026,1,2,14,30,tzinfo=dt.timezone.utc)
parent=ParentOrder("CONSTRUCTED-01","SIM-A",100.,start-dt.timedelta(minutes=1),start,100.,101.)
slots=[ExecutionSlot(i,start+dt.timedelta(minutes=i),start+dt.timedelta(minutes=i+1),start+dt.timedelta(minutes=i),start+dt.timedelta(minutes=i,seconds=30),200.,200.,200.,101.+i,observation_time=start-dt.timedelta(seconds=1),forecast_frozen_at=start-dt.timedelta(seconds=1)) for i in range(2)]
cfg=ExecutionConfig(commission_rate=.001,half_spread_bps=10.,temp_impact_coef=0.,perm_impact_coef=0.,max_participation=.2,cash_rate_annual=0.,borrow_rate_annual=0.)
result=execute_order(parent,"TWAP",slots,cfg,103.,start+dt.timedelta(minutes=3))
assert result.signed_filled_quantity==80. and result.signed_remainder_quantity==20.


In [ ]:
is_result=asdict(result.is_decomposition)
assert abs(is_result["identity_discrepancy"])<1e-10
assert abs(is_result["total_is_direct"]-sum(is_result[k] for k in ["delay_cost","realized_execution_cost","unfilled_cost","explicit_fees"]))<1e-10
assert is_result["delay_cost"]==100. and is_result["unfilled_cost"]==40.
print("CONSTRUCTED signed IS",is_result)
(RUN_DIR/"constructed_is.json").write_text(json.dumps({"data_kind":"CONSTRUCTED","IS":is_result},indent=2))
fig,ax=plt.subplots();keys=["delay_cost","realized_execution_cost","unfilled_cost","explicit_fees"];ax.bar(["Delay","Execution","Unfilled","Fees"],[is_result[k] for k in keys]);ax.set(ylabel="USD; positive = cost",title="CONSTRUCTED IS decomposition");fig.tight_layout();fig.savefig(RUN_DIR/"constructed_is.png");plt.close(fig)
